# Energy Minimization of Lennard-Jones charged particles — Conjugate Gradient

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

This notebook performs a simple **energy minimization (EM)** of a 2D system of
Lennard-Jones particles that may also carry a charge (Coulomb interaction).
Starting from a random arrangement, EM walks the system *downhill* on its potential-energy
surface towards a nearby local minimum. The minimizer here is a **conjugate-gradient**
algorithm (optionally preceded by a few **steepest-descent** steps) with an adaptive step
size.

## Theory in brief

### Lennard-Jones

$$E_{LJ}(r) \;=\; 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12}-\left(\frac{\sigma}{r}\right)^{6}\right]$$

The $(\sigma/r)^{12}$ term is the steep short-range **repulsion** (overlapping atoms), the
$-(\sigma/r)^{6}$ term the weaker long-range **attraction**. The two cancel exactly at
$r=\sigma$ (`Sigma`), and the energy reaches its minimum $-\varepsilon$ (`Epsilon`) a little
further out, at $R_{min}=2^{1/6}\sigma\approx1.12\,\sigma$. So `Sigma` sets the particle
*size* and `Epsilon` the well *depth*. The code evaluates this same expression from the
**squared** distance, so the inner loop needs no square root.

To save work, pairs beyond a cutoff (`CutOff`) are ignored — this makes the potential
slightly discontinuous at the cutoff, which is harmless here.

*(For the potential on its own — the two components plotted separately, in real force-field
units — see [`LJ-ELEC_Potentials`](LJ-ELEC_Potentials.ipynb).)*

### Coulomb

$$E_{Coul} = \frac{q_a q_b}{\epsilon_r\, r}$$

Like charges repel ($E>0$), unlike charges attract ($E<0$); the dielectric constant
`Dielec` ($\epsilon_r$) screens (weakens) the interaction.

### Minimization
The **force** on each atom is minus the gradient of the total energy, i.e. the local
downhill direction.

* **Steepest descent** moves every particle straight along the (normalised) total force
  $\mathbf{F}_k$. Simple, but it tends to *zig-zag* in long narrow valleys.
* **Conjugate gradient** follows a direction that mixes the current force with the previous
  search direction (Fletcher–Reeves), which cancels much of that zig-zag and usually
  converges in far fewer steps:

$$\mathbf{s}_k = \mathbf{F}_k + \gamma\, \mathbf{s}_{k-1}, \qquad
  \gamma = \frac{|\mathbf{F}_k|^2}{|\mathbf{F}_{k-1}|^2}$$

Particles move a step `dr` along $\mathbf{s}_k$, normalised as a single
$2N$-dimensional vector so that `dr` is the distance travelled by the whole configuration
(section 5). The first `numsteep` steps use plain steepest descent (with `numsteep = 0`, the
first step self-starts as steepest descent because $\gamma = 0$). The step is scaled **up** by `alpha` when the energy
decreases and **down** by `beta` when it increases; iteration stops when the energy change,
the step size, or the force norm drops below its threshold.

## 1. Imports

The numerical core uses only the Python **standard library** (`math`, `random`), so it runs
on a bare Python install. **matplotlib** is the one third-party dependency — it draws the
static figures and the trajectory animation (embedded inline as interactive HTML via
`jshtml`). The cell below first **installs matplotlib if it is missing** (handy on Google
Colab), then imports everything; `%matplotlib inline` renders figures inside the notebook.

In [ ]:
# --- Install required packages (works locally, on Colab, and on JupyterLite) ---
%pip install -q matplotlib

from math import sqrt

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from matplotlib import rc

from random import random, seed

# Show animations inline
rc('animation', html='jshtml')
%matplotlib inline

## 2. Helper functions

Small utilities used throughout:

* `dist` / `dist2` — Euclidean distance and its square (the squared form avoids a needless
  `sqrt` when we only need to compare distances).
* `SignR(a, b)` — returns `a` with the sign of `b`. It implements the **minimum-image
  convention**: the combination `tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)`
  wraps a coordinate difference into the range $[-\tfrac{box}{2}, +\tfrac{box}{2}]$, so each
  atom interacts with the *nearest periodic copy* of its neighbours (periodic boundary
  conditions).
* `charge_color` — purely cosmetic: white for positive charges, dark for negative, used when
  drawing the particles.

In [ ]:
### distance ###
def dist(A, B):
    return sqrt((A[0]-B[0])**2 + (A[1]-B[1])**2)

### squared distance ###
def dist2(A, B):
    return (A[0]-B[0])**2 + (A[1]-B[1])**2

### change sign ###
def SignR(a, b):
    if b > 0:
        return a
    else:
        return -a

### colour particles based on charge ###
def charge_color(charge, qat):
    if charge == qat:
        return "#FFFFFF"   # positive
    else:
        return "#333333"   # negative

## 3. Energy functions

Total energy is the sum over all particle pairs of the Lennard-Jones and Coulomb
contributions, using the **nearest image** convention (periodic boundary conditions).

Distances are handled as **squared** distances (`distsquare`) throughout the inner loops:
this avoids computing a `sqrt` for every pair, and lets the cutoff test (`distsquare <
cutoffsquare`) skip distant pairs cheaply. A `sqrt` is taken only where a term actually
needs $r$ itself (the Coulomb $1/r$).

In [ ]:
# LJ energy from the squared distance
def LJ2(distsquare, epsilon, sigma_exp6):
    # E_LJ = 4 eps [ (sigma/r)^12 - (sigma/r)^6 ], from the squared distance
    u = (1/distsquare)**3 * sigma_exp6        # u = (sigma/r)^6,  u*u = (sigma/r)^12
    return 4*epsilon * u * (u - 1)            # = 4 eps [ (sigma/r)^12 - (sigma/r)^6 ]

# classical Coulomb from the squared distance
def Coulomb2(r, dielec, qa, qb):
    return qa*qb / (dielec*sqrt(r))

# Calculate energy Evdw + Ecoulomb (uses squared distance), with periodic boundary conditions
def Calc_Ene2(coord, epsilon, sigma, dielec, cutoffsquare, boxdim, elec=1):
    Ene = 0.0
    ELJ = 0.0
    ECoul = 0.0
    sigma_exp6 = sigma**6
    # doubly nested loop over all particle pairs
    for i in range(len(coord)-1):
        for j in range(i+1, len(coord)):
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                vdw = LJ2(distsquare, epsilon, sigma_exp6)
                Ene += vdw
                ELJ += vdw
                if elec:
                    CC = Coulomb2(distsquare, dielec, qa, qb)
                    Ene += CC
                    ECoul += CC
    return Ene, ELJ, ECoul

## 4. Force functions

The force on an atom is minus the gradient of the total energy — the local *downhill*
direction. For the Lennard-Jones term the component along $x$ is obtained by the chain rule,

$$F_x = -\frac{\partial E}{\partial x} = -\frac{\partial E}{\partial u}\,
        \frac{\partial u}{\partial r}\,\frac{\partial r}{\partial x},
\qquad u=\left(\frac{\sigma}{r}\right)^{6},\quad E_{LJ}=4\varepsilon\,(u^{2}-u),$$

which maps directly onto the code:

* `dedu` $= \partial E/\partial u = 4\varepsilon\,(2u-1)$
* `dudr` $= \partial u/\partial r = -6\,\sigma^{6}/r^{7}$
* `drdx` $= x_i/r$, where `xi` $= x_j - x_i$ — using the $j-i$ difference already carries the
  sign that turns $-\partial E/\partial x_i$ into the force on atom $i$.

The Coulomb force follows the same pattern, with `dedr` $= -q_a q_b/(\epsilon_r\, r^{2})$.

In [ ]:
# LJ force component (uses squared distance)
def ForceLJ2(distsquare, epsilon, sigma_exp6, xi):
    # E_LJ = 4 eps [ (sigma/r)^12 - (sigma/r)^6 ] = 4 eps (u^2 - u) with u = (sigma/r)^6
    rij = sqrt(distsquare)
    u    = (1/distsquare)**3 * sigma_exp6
    dedu = 4*epsilon*(2*u - 1)                # dE/du
    dudr = sigma_exp6*(-6.0/rij**7.0)         # du/dr
    drdx = xi/rij
    return dedu*dudr*drdx

# Coulomb force component (uses squared distance)
def ForceCoulomb2(distsquare, dielec, qa, qb, xi):
    rij = sqrt(distsquare)
    dedr = -1.0*(qa*qb/dielec)*(1/distsquare)
    drdx = xi/rij
    return dedr*drdx

# Total force on each atom from Evdw + Ecoulomb (uses squared distance)
def Calc_Force2(coord, epsilon, sigma, dielec, cutoffsquare, boxdim):
    Force = []
    sigma_exp6 = sigma**6
    for i in range(len(coord)):
        tmpforce = [0.0, 0.0]
        for j in range(len(coord)):
            if i == j:
                continue
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                fflist = []
                for k in range(2):
                    tmp = coord[j][k] - coord[i][k]
                    ff = ForceLJ2(distsquare, epsilon, sigma_exp6, tmp)
                    ff += ForceCoulomb2(distsquare, dielec, qa, qb, tmp)
                    fflist.append(ff)
                for k in range(2):
                    tmpforce[k] = tmpforce[k] + fflist[k]
        Force.append(tmpforce)
    return Force

## 5. The minimizers: steepest descent & conjugate gradient

**Why two?** Steepest descent always steps along the local force, so in a long, narrow
energy valley it overshoots and *zig-zags*, wasting steps. Conjugate gradient corrects the
new direction with a fraction $\gamma$ of the previous one (here the **Fletcher–Reeves**
choice), keeping successive directions roughly *conjugate* and largely eliminating the
zig-zag — so it typically reaches the minimum in many fewer iterations.

Both functions take the current coordinates, the step size `drstep` and the forces, and
return the new coordinates, the force norm, and the **unit search directions** used
(`sdir`):

* `Steepest_descent` moves along the normalised total force.
* `Conjugate_gradient` builds the Fletcher–Reeves conjugate direction from the current
  force and the previous step's force/direction (`forceprev`, `sdirprev`), then normalises
  and moves along it.

Returning `sdir` (instead of using globals as the original GUI did) lets the run loop carry
the state it needs from one step to the next.

### What `dr` is the length of

A step needs a *direction* and a *length*, and both functions take the length from `drstep`. They
also agree on what that length is the length **of**: the search direction is normalised as **one
vector over the whole system** — all $2N$ coordinates at once — so `drstep` is the distance the
**configuration** moves through its $2N$-dimensional space, and the individual atoms share it out
in proportion to the force each one feels. An atom that is being squeezed moves a long way in a
step; one sitting comfortably in its well barely moves at all.

That is worth keeping in mind when reading step counts: one step is one force evaluation plus one
energy evaluation, and — because both steppers measure `dr` the same way — a step means the same
amount of movement whichever of the two took it.

In [ ]:
def Steepest_descent(atom_coord, drstep, force):
    """One steepest-descent step.

    Returns
    -------
    new_coord, normf, sdir
        sdir holds the per-atom components of the unit search direction
        (normalised over the whole system) actually used.
    """
    newlist = []
    sdir = []

    # 1) norm of the total force vector
    normf = 0.0
    for i in range(len(atom_coord)):
        normf = normf + force[i][0]**2.0 + force[i][1]**2.0
    normf = sqrt(normf)

    # 2) move every particle along the normalised force
    for i in range(len(atom_coord)):
        q = atom_coord[i][2]
        r0x = atom_coord[i][0]
        r0y = atom_coord[i][1]
        sx = sy = 0.0
        if normf > 0:
            sx = force[i][0]/normf
            sy = force[i][1]/normf
            r0x = r0x + drstep*sx
            r0y = r0y + drstep*sy
        sdir.append([sx, sy])
        newlist.append([r0x, r0y, q])
    return newlist, normf, sdir


def Conjugate_gradient(atom_coord, drstep, force, forceprev, sdirprev):
    """One conjugate-gradient (Fletcher-Reeves) step.

    Parameters
    ----------
    forceprev : forces from the previous step
    sdirprev  : unit search directions from the previous step

    Returns
    -------
    new_coord, normf, sdir
    """
    newlist = []
    sdir = []

    # 1) squared norms of the current and previous total force vectors
    normf2 = 0.0
    normf2prev = 0.0
    for i in range(len(atom_coord)):
        normf2 = normf2 + force[i][0]**2.0 + force[i][1]**2.0
        normf2prev = normf2prev + forceprev[i][0]**2.0 + forceprev[i][1]**2.0

    # Fletcher-Reeves coefficient (gamma = 0 on the first step -> pure steepest descent)
    if normf2prev > 0:
        gamma = normf2/normf2prev
    else:
        gamma = 0.0
    normf = sqrt(normf2)

    # 2) build the conjugate direction of the whole system:  s = F + gamma * s_prev
    svec = []
    for i in range(len(atom_coord)):
        svec.append([force[i][0] + gamma*sdirprev[i][0],
                     force[i][1] + gamma*sdirprev[i][1]])

    # 3) normalise it as ONE 2N-dimensional vector, exactly as Steepest_descent
    #    normalises the force, so that drstep is the distance the configuration moves
    norms = 0.0
    for i in range(len(atom_coord)):
        norms = norms + svec[i][0]**2.0 + svec[i][1]**2.0
    norms = sqrt(norms)

    # 4) move the particles along the normalised direction
    for i in range(len(atom_coord)):
        q = atom_coord[i][2]
        r0x = atom_coord[i][0]
        r0y = atom_coord[i][1]
        sx = sy = 0.0
        if norms > 0:
            sx = svec[i][0]/norms
            sy = svec[i][1]/norms
            r0x = r0x + drstep*sx
            r0y = r0y + drstep*sy
        sdir.append([sx, sy])
        newlist.append([r0x, r0y, q])
    return newlist, normf, sdir

## 6. Parameters

Change any of these parameters and re-run **this cell together with the Initialisation and Run cells just below** to explore their effect (or use *Kernel → Restart & Run All*). Values are in the
toy model's arbitrary units (lengths in box/canvas units, energies loosely in kcal/mol).

### System and its properties

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `nAtoms` | number of particles | 2–40 |
| `Radius` | particle radius (drawn size, and default absolute charge) | 10–40 — must leave room to place all atoms, or initialisation fails |
| `Sigma` | LJ distance parameter $\sigma$: the separation where $E_{LJ}=0$ (the minimum sits at $2^{1/6}\sigma$) | `2.24 * Radius` |
| `BoxDim` | box dimensions (periodic) | `[500, 500]` |
| `Epsilon` | LJ well depth (the minimum of $E_{LJ}$ is $-$`Epsilon`) | 0.25–25 |
| `Dielec` | dielectric constant (charge screening) | 1 (vacuum) – 80 (water) |
| `qat` | absolute charge per atom | defaults to `Radius` |
| `frac_neg` | fraction of negative charges | 0–1 |
| `CutOff` | non-bonded cutoff distance | 250 |

### Minimizer

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `drinit` | initial step size `dr` | 0.1–5 |
| `drmin` / `drmax` | min / max allowed `dr` | 1e-5 / 5 |
| `alpha` / `beta` | scale `dr` up / down after each step | 1.05 / 0.90 |
| `deltaE` | energy-change stop threshold (tighter -> settles into a deeper minimum) | 1e-5 |
| `normFmin` | force-norm stop threshold | 1e-4 |
| `numsteep` | initial steepest-descent steps before conjugate gradient | 0+ |
| `max_iter` | hard cap on the number of steps | 2000 |

In [ ]:
nAtoms  = 20              # number of atoms
Radius  = 25.0            # atom radius (must be in a sensible range vs nAtoms so all atoms fit)
Sigma   = 2.24 * Radius   # LJ distance parameter sigma (E_LJ = 0 here; minimum at 2^(1/6)*Sigma)
BoxDim  = [500, 500]      # box dimensions
Epsilon = 6.25            # LJ well depth (the minimum of E_LJ is -Epsilon)
Dielec  = 1.0             # dielectric constant
qat     = Radius          # atom absolute charge
frac_neg = 0.5            # fraction of negative charges
OverlapFr = 0.0           # fraction of overlap allowed when placing atoms
CutOff  = 250             # non-bonded cutoff
CutOffSquare = CutOff**2

# --- minimizer controls ---
drinit  = 1.00        # initial dr for EM
drmin   = 0.00001     # minimum dr value to keep stepping
drmax   = 5.00        # maximum dr
alpha   = 1.05        # scale factor for dr when Enew < Eold
beta    = 0.90        # scale factor for dr when Enew > Eold
deltaE  = 0.00001     # energy-difference threshold to stop EM (tight -> avoids stopping early on a plateau)
normFmin = 0.0001     # minimum force norm to keep stepping

numsteep = 0          # number of initial steepest-descent steps before conjugate gradient
Seed    = 100         # random number seed
max_iter = 2000       # safety cap on the number of EM steps

## 7. Initialisation

Generate random, non-overlapping starting positions and assign charges
(a fraction `frac_neg` negative, the rest positive).

In [ ]:
import sys

### generate random, non-overlapping coordinates ###
def InitConf(n, dim, radius, qat, frac_neg):
    seed(Seed)
    print("Initializing box, please wait...")
    tmp_coord = []
    i = 0
    ntrial = 0
    nneg = int(float(n) * frac_neg)
    npos = n - nneg

    # first atom
    x = random()*(dim[0]-2*radius) + radius
    y = random()*(dim[1]-2*radius) + radius
    charge = -qat
    if npos == n:
        charge = qat
    i += 1
    if n == 2:
        tmp_coord.append([175, 300, charge])
    else:
        tmp_coord.append([x, y, charge])

    # remaining negative charges
    while i < nneg:
        x = random()*(dim[0]-2*radius) + radius
        y = random()*(dim[1]-2*radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            charge = -qat
            if n == 2:
                tmp_coord.append([325, 300, charge])
            else:
                tmp_coord.append([x, y, charge])
            i += 1
        ntrial += 1
        if ntrial > 100000:
            print("initialisation failed")
            print("==> reduce radius or number of atoms")
            sys.exit()

    # remaining positive charges
    while i < n:
        x = random()*(dim[0]-2*radius) + radius
        y = random()*(dim[1]-2*radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            charge = qat
            if n == 2:
                tmp_coord.append([325, 300, charge])
            else:
                tmp_coord.append([x, y, charge])
            i += 1
        ntrial += 1
        if ntrial > 10**10:
            print("initialisation failed")
            print("==> reduce radius or number of atoms")
            sys.exit()
    return tmp_coord


Atom_Coord = InitConf(nAtoms, BoxDim, Radius, qat, frac_neg)
Color = [charge_color(a[2], qat) for a in Atom_Coord]
print(f"Placed {len(Atom_Coord)} atoms.")

## 8. Run the minimization

At each step the minimizer computes
the forces, takes a minimization step — **steepest descent for the first `numsteep` steps
(and the very first step), conjugate gradient thereafter** — adaptively rescales `dr`,
applies periodic boundary conditions, and records the trajectory and energies. It carries
the previous force and search direction (`forceprev`, `sdirprev`) forward for the conjugate-
gradient update.

It stops as soon as **any** of three convergence criteria is met — the energy change falls
below `deltaE`, the step size drops below `drmin`, or the average force norm falls below
`normFmin` — or after at most `max_iter` steps.

In [ ]:
def run_minimization(Atom_Coord):
    """Headless minimization: `numsteep` steepest-descent steps, then conjugate gradient.
    Returns trajectory + energy history."""
    coord = [list(a) for a in Atom_Coord]   # work on a copy
    drstep = drinit

    # force / search-direction state carried between steps
    forceprev = [[0.0, 0.0] for _ in coord]
    sdirprev  = [[0.0, 0.0] for _ in coord]

    Ene, EneLJ, EneCoul = Calc_Ene2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
    Ene_prev = Ene

    traj    = [ [list(a) for a in coord] ]   # snapshot per step
    E_hist  = [Ene]
    Elj_hist = [EneLJ]
    Ecoul_hist = [EneCoul]

    print("Iteration: %8d Epot: %6.1f Elj: %6.1f Ecoul: %6.1f" % (0, Ene, EneLJ, EneCoul))

    for step in range(1, max_iter+1):
        Force = Calc_Force2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)

        # steepest descent for the first `numsteep` steps (and the very first step),
        # conjugate gradient afterwards
        if step <= numsteep or step == 1:
            coord, normF, sdir = Steepest_descent(coord, drstep, Force)
        else:
            coord, normF, sdir = Conjugate_gradient(coord, drstep, Force, forceprev, sdirprev)

        Ene, EneLJ, EneCoul = Calc_Ene2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
        Ene_diff = Ene - Ene_prev

        # adaptive step size
        if Ene_diff < 0.0:
            drstep = min(drmax, drstep*alpha)
        else:
            drstep = drstep*beta
        Ene_prev = Ene

        # remember this step's force and search direction for the next CG step
        forceprev = Force
        sdirprev = sdir

        # periodic boundary conditions
        for pp in range(len(coord)):
            for k in range(2):
                if coord[pp][k] < 0:
                    coord[pp][k] += BoxDim[k]
                if coord[pp][k] > BoxDim[k]:
                    coord[pp][k] -= BoxDim[k]

        normF = normF/len(coord)

        traj.append([list(a) for a in coord])
        E_hist.append(Ene)
        Elj_hist.append(EneLJ)
        Ecoul_hist.append(EneCoul)

        if step % 50 == 0:
            print("Iteration: %8d Epot: %6.1f Elj: %6.1f Ecoul: %6.1f deltaE: %10.6f <normF>: %8.6f dr: %8.6f"
                  % (step, Ene, EneLJ, EneCoul, Ene_diff, normF, drstep))

        # convergence
        if abs(Ene_diff) < deltaE or drstep < drmin or normF < normFmin:
            print("STOPPING... deltaE<%g, or drstep<%g, or normF<%g" % (deltaE, drmin, normFmin))
            print("Iteration: %8d Epot: %6.1f Elj: %6.1f Ecoul: %6.1f deltaE: %10.6f <normF>: %8.6f dr: %8.6f"
                  % (step, Ene, EneLJ, EneCoul, Ene_diff, normF, drstep))
            break

    return traj, E_hist, Elj_hist, Ecoul_hist


traj, E_hist, Elj_hist, Ecoul_hist = run_minimization(Atom_Coord)
print(f"\nDone in {len(traj)-1} steps. Final Epot = {E_hist[-1]:.2f}")

## 9. Energy convergence

How the total, Lennard-Jones and Coulomb energies evolve during the minimization.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
steps = range(len(E_hist))
ax.plot(steps, E_hist,     label="Epot (total)", lw=2)
ax.plot(steps, Elj_hist,   label="E$_{LJ}$",  lw=1.5)
ax.plot(steps, Ecoul_hist, label="E$_{Coul}$", lw=1.5)
ax.set_xlabel("EM step")
ax.set_ylabel("Energy (kcal/mol)")
ax.set_title("Steepest-descent energy minimization")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 10. Visualise the system

Start and end configurations side by side. White = positive charge, dark = negative.

In [ ]:
def draw_config(ax, coord, title):
    ax.set_xlim(0, BoxDim[0])
    ax.set_ylim(0, BoxDim[1])
    ax.set_aspect('equal')
    ax.set_facecolor("#ccddff")
    ax.set_title(title)
    ax.invert_yaxis()   # match the original canvas (y downwards)
    for a in coord:
        col = charge_color(a[2], qat)
        ax.add_patch(Circle((a[0], a[1]), Radius, facecolor=col, edgecolor="black", lw=0.8))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5.5))
draw_config(ax1, traj[0],  f"Initial  (Epot = {E_hist[0]:.1f})")
draw_config(ax2, traj[-1], f"Minimized (Epot = {E_hist[-1]:.1f})")
plt.tight_layout()
plt.show()

## 11. Animation of the minimization

Replays the whole trajectory.
(To keep it light, only every few frames are shown — adjust `stride`.)

In [ ]:
stride = max(1, len(traj)//120)   # cap at ~120 frames
frames = list(range(0, len(traj), stride))

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, BoxDim[0])
ax.set_ylim(0, BoxDim[1])
ax.set_aspect('equal')
ax.set_facecolor("#ccddff")
ax.invert_yaxis()

circles = [Circle((a[0], a[1]), Radius,
                  facecolor=charge_color(a[2], qat), edgecolor="black", lw=0.8)
           for a in traj[0]]
for c in circles:
    ax.add_patch(c)
title = ax.set_title("")

def update(frame_idx):
    f = frames[frame_idx]
    for c, a in zip(circles, traj[f]):
        c.center = (a[0], a[1])
    title.set_text(f"step {f}   Epot = {E_hist[f]:.1f}")
    return circles + [title]

anim = FuncAnimation(fig, update, frames=len(frames), interval=80, blit=False)
plt.close(fig)   # avoid a duplicate static figure
anim

## 12. Comparison of the three minimizers

This notebook implements **the conjugate-gradient minimizer (with an optional steepest-descent
warm-up)**. The two companion notebooks minimize the **same system** — 20 particles with identical
parameters and the same random seed (`Seed = 100`). Running each **with its default parameters**
gives:

| Method | Notebook | Final energy | Steps |
|---|---|---|---|
| Steepest descent | [`LJ-ELEC_EM-steepest`](LJ-ELEC_EM-steepest.ipynb) | ≈ −409 | ~1110 |
| Conjugate gradient | this one | ≈ −409 | **~590** |
| Simplex (Nelder–Mead) | [`LJ-ELEC_EM-simplex`](LJ-ELEC_EM-simplex.ipynb) | ≈ −187 | ~620 |

Take-aways:

* All three reach a **local** minimum — none is guaranteed to find the global minimum, and the
  result depends on the starting configuration (the random seed) and the path taken.
* **Conjugate gradient** does here what it promises: it reaches **the same minimum as steepest
  descent in about half the steps**. Averaged over thirty different starting configurations it is
  also about 26 kcal/mol deeper and ~200 steps cheaper — the memory term really does damp the
  steepest-descent zig-zag.
* The **simplex** is *derivative-free* (energy only, no forces) — simple and robust, but it scales
  poorly to this 40-dimensional search space (2 × 20 coordinates), so it converges to a much
  shallower minimum. A good illustration of why gradient-based methods dominate for smooth,
  high-dimensional problems.

*(Energies and step counts are for `Seed = 100` with each notebook's default parameters. One seed is
one sample: the exercise below shows how much these numbers move when the starting configuration
changes.)*

---
## **13. Exercise**

Everything above ran the conjugate gradient with `numsteep = 0` — no steepest-descent warm-up at
all. The exercise below asks whether that was the right choice.

> **How it works.** The exercise starts from a stub containing a `TODO` placeholder. Replace it
> with your own code and re-run the cell. The cells are **self-checking**: they stay quiet until
> your code works, then fill themselves in with the numbers and plots you need. A worked solution
> is folded away at the end — try it yourself first.

Run the cell below once before starting. It wraps the run loop of section 8 in a function,
`run_em(...)`, whose parameters are **arguments instead of globals**, so the exercise can launch
dozens of minimizations without editing section 6. It is the same algorithm — same forces, same
two steppers, same adaptive `dr`, same convergence test — only quieter, and with the rule that
decides *which* stepper to use at a given step handed in as an argument (that is what you will
write). `config_for_seed` builds a fresh starting configuration for a given random seed.

In [ ]:
import io, contextlib, statistics
from math import sqrt

def run_em(numsteep=0, use_steepest=None, drinit=drinit, drmax=drmax, alpha=alpha, beta=beta,
           deltaE=deltaE, drmin=drmin, normFmin=normFmin, max_iter=max_iter,
           coord0=None, keep_traj=False):
    """The run loop of section 8, with the parameters as arguments instead of globals.

    Same algorithm: forces -> one minimization step -> rescale dr -> periodic boundaries ->
    convergence test.  Two differences: it prints nothing, and the rule that decides which
    stepper a given step uses is supplied as `use_steepest(step, numsteep)` (the exercise
    below asks you to write it; None falls back to the rule of section 8).

    Returns a dict with  E (final energy), steps, E_hist, coord (final),
    traj (only if keep_traj=True) and reason ('converged' / 'max_iter').
    """
    if use_steepest is None:
        use_steepest = lambda step, numsteep: step <= numsteep or step == 1

    coord = [list(a) for a in (Atom_Coord if coord0 is None else coord0)]
    drstep = drinit
    forceprev = [[0.0, 0.0] for _ in coord]
    sdirprev  = [[0.0, 0.0] for _ in coord]

    Ene, _, _ = Calc_Ene2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
    Ene_prev = Ene
    E_hist = [Ene]
    traj = [[list(a) for a in coord]] if keep_traj else None
    reason, steps, nsteep = 'max_iter', 0, 0

    for step in range(1, max_iter+1):
        Force = Calc_Force2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
        if use_steepest(step, numsteep):
            coord, normF, sdir = Steepest_descent(coord, drstep, Force)
            nsteep += 1
        else:
            coord, normF, sdir = Conjugate_gradient(coord, drstep, Force, forceprev, sdirprev)
        Ene, _, _ = Calc_Ene2(coord, Epsilon, Sigma, Dielec, CutOffSquare, BoxDim)
        Ene_diff = Ene - Ene_prev

        if Ene_diff < 0.0:
            drstep = min(drmax, drstep*alpha)
        else:
            drstep = drstep*beta
        Ene_prev = Ene
        forceprev, sdirprev = Force, sdir

        for pp in range(len(coord)):
            for k in range(2):
                if coord[pp][k] < 0:
                    coord[pp][k] += BoxDim[k]
                if coord[pp][k] > BoxDim[k]:
                    coord[pp][k] -= BoxDim[k]

        steps = step
        E_hist.append(Ene)
        if keep_traj:
            traj.append([list(a) for a in coord])

        if abs(Ene_diff) < deltaE or drstep < drmin or normF/len(coord) < normFmin:
            reason = 'converged'
            break

    return {'E': E_hist[-1], 'steps': steps, 'E_hist': E_hist, 'coord': coord,
            'traj': traj, 'reason': reason, 'nsteep': nsteep, 'numsteep': numsteep}


def quiet(f, *args, **kw):
    """Call f while swallowing InitConf's progress messages."""
    with contextlib.redirect_stdout(io.StringIO()):
        return f(*args, **kw)


def config_for_seed(s):
    """A fresh random starting configuration for random seed `s`."""
    global Seed
    Seed = s
    return quiet(InitConf, nAtoms, BoxDim, Radius, qat, frac_neg)


print("toolbox ready -- reference run (numsteep = %d): " % numsteep, end="")
_ref = run_em(numsteep=numsteep)
print(f"E = {_ref['E']:.2f} in {_ref['steps']} steps ({_ref['reason']})")

---
### **Exercise — warm up with steepest descent?**

Section 8 contains the switch: the first `numsteep` steps use **steepest descent**, the rest
**conjugate gradient**. The notebook ships with `numsteep = 0`, so how much a warm-up helps has
never been measured. Do it.

There is a real argument on each side. **For:** the Fletcher–Reeves direction is built from the
*previous* direction, and its theory assumes the surface is roughly quadratic — which it is near a
minimum, but certainly not at a random starting configuration where a few atoms nearly overlap and
the forces are enormous and about to change completely. Carrying that history forward may be worse
than useless. **Against:** the whole point of conjugate gradient is to avoid the zig-zag that
steepest descent spends its steps on; deliberately taking steepest-descent steps throws that away.

**(a)** Decide which argument you believe, and write down a prediction: will a warm-up give a
**deeper** minimum, a **faster** one, both, or neither? How long a warm-up would you try?

**(b)** Write `use_steepest(step, numsteep)` in the cell below — the rule the run loop of section 8
uses to choose a stepper. Mind the very first step: there is no previous search direction for
$\gamma$ to use.

**(c)** The cell then sweeps `numsteep` from 0 up to `max_iter`. Read the table and the first two
panels of the figure. Does the warm-up change **where** the minimizer ends up, or only **how long**
it takes to get there? Which `numsteep` is fastest, and what happens once the warm-up gets long?
(And why do `numsteep = 0` and `numsteep = 1` give exactly the same run?)

**(d)** The last row sets `numsteep = max_iter`, so the run never switches: the notebook becomes a
**pure steepest-descent** minimizer. Compare it with the steepest-descent row of the table in
section 12, and with the [steepest-descent notebook](LJ-ELEC_EM-steepest.ipynb) itself. Do they
agree *exactly*? Why is that worth checking before you trust any of the numbers above?

**(e)** A single starting configuration cannot separate luck from skill: the same system minimized
from a different random start lands in a different local minimum, tens of kcal/mol away (the
[steepest-descent notebook](LJ-ELEC_EM-steepest.ipynb) measures exactly that in its own Exercise 3).
The last cell therefore repeats the comparison over **ten** starting configurations and prints, for
each warm-up length, the **paired** difference against `numsteep = 0` with its standard error. Does
the warm-up survive? Would you call the effect established? And which difference in that table
*is* real?

**(f)** Write the one-sentence answer you would give a colleague who asks "should I warm up my
conjugate-gradient minimisation with a few steepest-descent steps?" Then one more thought: the
conjugate-gradient method is derived assuming every step is an **exact line search** along the
search direction, while this notebook merely multiplies `dr` by `alpha` or `beta`. How much of the
conjugacy do you expect to survive that?

In [ ]:
# ---------- Exercise: which steps should be steepest descent? ----------
TODO = None

def use_steepest(step, numsteep):
    """Return True if step number `step` (counting from 1) should be a steepest-descent
    step rather than a conjugate-gradient one, given a warm-up length `numsteep`.

    One line -- read the rule in the run loop of section 8, and mind the very first step.
    """
    return TODO                                    # <-- your code here


# ---------- self-check ----------
_cases = [((1, 0), True,  "step 1 with no warm-up  (nothing to build a conjugate direction from)"),
          ((2, 0), False, "step 2 with no warm-up  (conjugate gradient from here on)"),
          ((5, 10), True, "step 5 of a 10-step warm-up"),
          ((10, 10), True, "last step of a 10-step warm-up"),
          ((11, 10), False, "first step after the warm-up")]

def solved_rule():
    try:
        return all(use_steepest(*args) == want for args, want, _ in _cases[1:])
    except Exception:
        return False

if not solved_rule():
    sweep = None
    print("Not yet: replace the TODO in use_steepest above and re-run this cell.")
    try:
        for args, want, what in _cases:
            got = use_steepest(*args)
            print(f"   use_steepest{args} = {got!s:<6} expected {want!s:<6} {what}")
    except Exception as exc:
        print(f"   ({type(exc).__name__}: {exc})")
else:
    for args, want, what in _cases:
        got = "OK" if use_steepest(*args) == want else "<-- not the rule of section 8"
        print(f"  use_steepest{str(args):<9} = {str(use_steepest(*args)):<6} {got:<30} {what}")
    if use_steepest(1, 0) is not True:
        print("\n  Note: you left out the `step == 1` case. Part (e) shows why it is not just "
              "cosmetic here.")

    # ---------- (c) the warm-up sweep ----------
    NUMSTEEP = [0, 1, 2, 5, 10, 20, 50, 100, 200, 500, max_iter]
    print(f"\nminimizing the same starting configuration (E = {E_hist[0]:.2f}) "
          f"with warm-ups of {NUMSTEEP}\n")
    print(f"{'numsteep':>9} {'E after warm-up':>16} {'final E':>10} {'steps':>7}  {'SD steps':>9}  stopped by")
    sweep = []
    for ns in NUMSTEEP:
        r = run_em(numsteep=ns, use_steepest=use_steepest)
        r['E_warm'] = r['E_hist'][min(ns, len(r['E_hist'])-1)]
        sweep.append(r)
        tag = "   <-- the notebook's default" if ns == numsteep else (
              "   <-- pure steepest descent" if ns >= max_iter else "")
        print(f"{ns:9d} {r['E_warm']:16.2f} {r['E']:10.2f} {r['steps']:7d}  {r['nsteep']:9d}  "
              f"{r['reason']}{tag}")

    base = sweep[0]
    best_E = min(sweep, key=lambda r: r['E'])
    best_n = min(sweep, key=lambda r: r['steps'])
    print(f"\n  no warm-up      : E = {base['E']:.2f} in {base['steps']} steps")
    print(f"  deepest         : E = {best_E['E']:.2f} in {best_E['steps']} steps "
          f"(numsteep = {best_E['numsteep']})")
    print(f"  fewest steps    : E = {best_n['E']:.2f} in {best_n['steps']} steps "
          f"(numsteep = {best_n['numsteep']})")
    print(f"  spread over the sweep: {max(r['E'] for r in sweep) - min(r['E'] for r in sweep):.1f} "
          f"kcal/mol, {min(r['steps'] for r in sweep)}-{max(r['steps'] for r in sweep)} steps")

In [ ]:
# What the warm-up does: depth, cost, and the first few hundred steps
if sweep is None:
    print("Complete use_steepest in the cell above to get the figure.")
else:
    ns_vals = [r['numsteep'] for r in sweep]
    xs = [max(n, 0.5) for n in ns_vals]          # 0 -> 0.5 so it can sit on a log axis
    pure_sd, pure_cg = sweep[-1], sweep[0]

    fig, (axE, axN, axC) = plt.subplots(1, 3, figsize=(14.5, 4.6))

    axE.semilogx(xs, [r['E'] for r in sweep], "o-", color="#6a3d9a", lw=1.6, ms=7)
    axE.axhline(pure_cg['E'], color="#1f6fb4", ls="--", lw=1.2)
    axE.axhline(pure_sd['E'], color="#1b8a5a", ls="--", lw=1.2)
    axE.text(0.02, pure_cg['E'], "no warm-up (pure CG)", transform=axE.get_yaxis_transform(),
             ha="left", va="bottom", color="#1f6fb4", fontsize=8)
    axE.text(0.98, pure_sd['E'], " pure steepest descent", transform=axE.get_yaxis_transform(),
             ha="right", va="top", color="#1b8a5a", fontsize=8)
    axE.set_xlabel("numsteep  (0 plotted at 0.5)")
    axE.set_ylabel("final Epot (kcal/mol)")
    axE.set_title("does a warm-up find a deeper minimum?", fontsize=10)
    axE.grid(alpha=0.3, which="both")

    axN.semilogx(xs, [r['steps'] for r in sweep], "o-", color="#d1701a", lw=1.6, ms=7)
    axN.axhline(pure_cg['steps'], color="#1f6fb4", ls="--", lw=1.2)
    axN.axhline(pure_sd['steps'], color="#1b8a5a", ls="--", lw=1.2)
    axN.set_xlabel("numsteep  (0 plotted at 0.5)")
    axN.set_ylabel("steps to convergence")
    axN.set_title("... or reach it faster?", fontsize=10)
    axN.grid(alpha=0.3, which="both")

    nshow = 400
    axC.plot(range(min(nshow, len(pure_cg['E_hist']))), pure_cg['E_hist'][:nshow],
             color="#1f6fb4", lw=1.8, label="pure conjugate gradient")
    axC.plot(range(min(nshow, len(pure_sd['E_hist']))), pure_sd['E_hist'][:nshow],
             color="#1b8a5a", lw=1.8, label="pure steepest descent")
    axC.set_xlabel("EM step")
    axC.set_ylabel("Epot (kcal/mol)")
    axC.set_title("which descends faster per step?\n(first %d steps -- but see the step sizes below)" % nshow,
                  fontsize=10)
    axC.legend(fontsize=8, loc="upper right")
    axC.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# ---------- (f) the same comparison over ten starting configurations ----------
if sweep is None:
    print("Complete use_steepest in the cell above to get this comparison.")
else:
    SEEDS = list(range(100, 110))
    VARIANTS = [0, 10, 50, max_iter]               # max_iter = pure steepest descent
    _Seed_notebook = Seed

    print(f"minimizing {len(SEEDS)} starting configurations with numsteep = {VARIANTS} "
          f"({len(SEEDS)*len(VARIANTS)} runs, this takes a moment)...\n")
    table = {ns: [] for ns in VARIANTS}
    for s in SEEDS:
        coord = config_for_seed(s)
        for ns in VARIANTS:
            table[ns].append(run_em(numsteep=ns, use_steepest=use_steepest, coord0=coord))
    Seed = _Seed_notebook                          # leave the notebook's own seed as we found it

    head = f"{'seed':>5} " + " ".join(f"{('numsteep=%d' % ns):>19}" for ns in VARIANTS)
    print(head)
    for k, s in enumerate(SEEDS):
        print(f"{s:5d} " + " ".join(f"{table[ns][k]['E']:11.2f} /{table[ns][k]['steps']:6d}"
                                    for ns in VARIANTS))

    print(f"\n{'numsteep':>9} {'mean E':>9} {'sd':>7} {'best E':>9} {'mean steps':>11}")
    for ns in VARIANTS:
        E = [r['E'] for r in table[ns]]
        n = [r['steps'] for r in table[ns]]
        print(f"{ns:9d} {statistics.mean(E):9.1f} {statistics.stdev(E):7.1f} {min(E):9.1f} "
              f"{statistics.mean(n):11.1f}")

    print("\npaired against numsteep = 0, per starting configuration "
          "(negative = the warm-up found a deeper minimum):")
    for ns in VARIANTS[1:]:
        d = [a['E'] - b['E'] for a, b in zip(table[ns], table[0])]
        sem = statistics.stdev(d)/sqrt(len(d))
        print(f"  numsteep = {ns:5d}:  mean difference {statistics.mean(d):+7.1f} "
              f"+/- {sem:.1f} (s.e.m.), deeper in {sum(1 for x in d if x < 0)}/{len(d)} runs")

In [ ]:
# Every starting configuration, every warm-up: does any column win consistently?
if sweep is None:
    print("Complete use_steepest in the cell above to get the figure.")
else:
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    xpos = range(len(VARIANTS))
    labels = [("pure SD" if ns >= max_iter else str(ns)) for ns in VARIANTS]

    for k, s in enumerate(SEEDS):
        ys = [table[ns][k]['E'] for ns in VARIANTS]
        ax.plot(xpos, ys, "o-", color="#b0a8c0", lw=1.0, ms=4, zorder=1)
        ax.annotate(str(s), xy=(0, ys[0]), xytext=(-6, 0), textcoords="offset points",
                    ha="right", va="center", fontsize=7, color="#777777")
    cols = [[table[ns][k]['E'] for k in range(len(SEEDS))] for ns in VARIANTS]
    means = [statistics.mean(col) for col in cols]
    sems = [statistics.stdev(col)/sqrt(len(col)) for col in cols]
    ax.errorbar(list(xpos), means, yerr=sems, fmt="o-", color="#b03030", lw=2.6, ms=9,
                capsize=5, zorder=3, label="mean $\\pm$ s.e.m.")

    ax.set_xticks(list(xpos))
    ax.set_xticklabels(labels)
    ax.set_xlabel("numsteep  (steepest-descent steps before conjugate gradient)")
    ax.set_ylabel("final Epot (kcal/mol)")
    ax.set_title("one line per starting configuration\n"
                 "flat across the warm-ups; only dropping conjugate gradient moves it",
                 fontsize=10)
    ax.margins(x=0.12)
    ax.grid(alpha=0.3, axis="y")
    ax.legend(fontsize=9, loc="upper left", framealpha=0.95)
    plt.tight_layout()
    plt.show()

<details>
<summary><b>Worked solution and expected answers</b></summary>

**(b)**

```python
def use_steepest(step, numsteep):
    return step <= numsteep or step == 1
```

**(c)** Sweeping the warm-up on the notebook's own starting configuration (`Seed = 100`):

| `numsteep` | final E | steps |
|---|---|---|
| 0 | −409.44 | 589 |
| 1 | −409.44 | 589 |
| 2 | −409.44 | 638 |
| 5 | −393.79 | **530** |
| 10 | −409.44 | 610 |
| 20 | −409.43 | 587 |
| 50 | −409.44 | 656 |
| 100 | −409.43 | 853 |
| 200 | −409.44 | 764 |
| 500 | −409.44 | 969 |
| 2000 (= `max_iter`) | −409.45 | 1109 |

The **energy column is flat**: every warm-up length but one lands in the same minimum, −409.4.
Whatever the warm-up does, it does not change *where* this system ends up.

What does change is the **cost**, and beyond `numsteep ≈ 20` it changes in one direction only: the
longer the warm-up, the more steps, sliding from 589 towards the 1109 of pure steepest descent
(656, 853, 969 …). That is exactly what you would expect from replacing an efficient stepper by a
less efficient one for longer and longer. Short warm-ups buy nothing either: 2, 10 and 20 steps of
steepest descent cost 638, 610 and 587 steps against the 589 of no warm-up — differences of a
couple of per cent, in both directions. The one entry that looks fast, `numsteep = 5` at 530 steps,
is also the one run that ends somewhere else entirely (−393.79 instead of −409.4): it is not a
quicker route to the same place, it is a different place. These runs are chaotic, and nearly all of
the 15.7 kcal/mol "spread" printed by the cell comes from that single row.

`numsteep = 0` and `numsteep = 1` are identical because of the `step == 1` clause: step 1 is a
steepest-descent step either way.

**(d)** Yes, **exactly**: −409.45 in 1109 steps, the same numbers to the last decimal as the
steepest-descent notebook. Both notebooks share the same energy, force, initialisation and run-loop
code, so if they disagreed, one of them had a bug rather than a result. Whenever a notebook can be
reduced to another one by a parameter, check that it reproduces it.

**(e)** Over seeds 100–109:

| `numsteep` | mean final E | sd | mean steps |
|---|---|---|---|
| 0 | −363.6 | 27.9 | 766 |
| 10 | −366.0 | 29.5 | 637 |
| 50 | −361.0 | 33.4 | 744 |
| `max_iter` (pure SD) | −343.4 | 30.1 | 941 |

The paired differences against `numsteep = 0` are −2.4 ± 4.7 (deeper in 6 of 10 runs) for
`numsteep = 10` and +2.6 ± 8.6 (5 of 10) for `numsteep = 50`. Both are smaller than their own error
bars: **nothing is established**. Thirty starting configurations confirm it — `numsteep = 10`
versus 0 gives **+6.1 ± 4.3** in energy (deeper in 14/30). Only the step count keeps a hint of an
effect, −77 ± 35 (fewer in 17/30), and at barely two standard errors that is not something to build
a recommendation on.

The difference in that table that *is* real is the last row: pure steepest descent ends
**+20.2 ± 9.0** higher than the conjugate gradient over these ten seeds, and needs about 45 % more
steps. Over thirty seeds the same comparison gives −25.7 ± 5.8 in favour of conjugate gradient, with
−195 ± 85 steps — deeper *and* cheaper, which is the effect worth having. The warm-up is not one.

**(f)** *"No: a steepest-descent warm-up leaves this minimizer where it was going anyway, and any
change in cost is inside the run-to-run noise — so keep `numsteep = 0` unless you can show
otherwise on your own problem, with many starts."*

As for conjugacy: not all of it survives. Fletcher–Reeves is derived for an **exact line search** —
you are supposed to reach the minimum *along* each direction before building the next one, which is
what makes successive directions conjugate and gives the method its "quadratic function in N steps"
property. Here `dr` is simply multiplied by 1.05 or 0.9, so each step lands wherever it lands and
the directions are only loosely conjugate. What survives is still worth having — roughly a quarter
fewer steps and ~26 kcal/mol of depth against steepest descent — and a real line search would be
the way to find out how much more is on the table.

</details>

</details>

---
### **Going further**

Open-ended, no scaffolding provided:

* **A real line search.** Replace the `alpha`/`beta` rescaling with an actual search along the
  current direction — try three step sizes and fit a parabola, say. Now the directions are much
  closer to conjugate: how much more does conjugate gradient gain, and does a warm-up start to
  matter?
* **Polak–Ribière instead of Fletcher–Reeves.** Use
  $\gamma = \mathbf{F}_k\cdot(\mathbf{F}_k-\mathbf{F}_{k-1})/|\mathbf{F}_{k-1}|^2$, which
  automatically resets itself ($\gamma \approx 0$) when successive forces stop resembling each
  other. Does it beat Fletcher–Reeves here?
* **Restart on a schedule.** A standard cure for a stale conjugate direction is to throw it away
  every $N$ steps and take a plain steepest-descent step. That is a warm-up repeated for ever —
  sweep $N$ and see whether *periodic* restarting does what an initial warm-up could not.
* **Give the memory its magnitude back.** `sdirprev` is stored as a *unit* vector, so the
  $\gamma\,\mathbf{s}_{k-1}$ term is mixed with a force in different units. Try keeping the
  unnormalised direction instead and normalising only when stepping — textbook conjugate gradient.
  (It diverges with this adaptive-`dr` scheme; work out why before concluding the textbook is
  wrong.)